# RuleShift Notebook

## Runtime bootstrap

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/ruleshift-runtime/src — "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

## Frozen leaderboard split

In [ ]:
from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)
PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe()

## Official task

In [ ]:
@kbench.task(
    name="ruleshift_benchmark_v1_binary_row",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
    store_task=False,
)
def _ruleshift_benchmark_v1_binary_row(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> dict:
    import json

    eval_df = leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]].copy()
    binary_results = _ruleshift_benchmark_v1_binary_row.evaluate(
        llm=[llm],
        evaluation_data=eval_df,
    )
    binary_df = normalize_count_result_df(
        binary_results.as_dataframe(),
        dataframe_label="binary_df",
    )
    payload = build_kaggle_payload(binary_df=binary_df)
    validate_kaggle_payload(payload)

    print("=== RuleShift Notebook — Canonical Result ===")
    print(json.dumps(payload, indent=2))
    print("=== End Canonical Result ===")
    return payload


## Official payload

In [ ]:
payload = ruleshift_benchmark_v1_binary(kbench.llm)


## Task selection

In [ ]:
%choose ruleshift_benchmark_v1_binary